**Machine Learning Analysis: Gut Microbiome Classification**

The development and assessment of machine learning models to classify patient samples into three clinical states (Alzheimer's Disease, Mild Cognitive Impairment, Healthy Control)

By using microbiome-derived feature sets that contain these two types of Feature Sets (Relative Abundance Features and Relative Abundances plus MICOM Prediction Flux Feature Sets).

In [ ]:
# Machine Learning Analysis Notebook:
# This notebook is used to show the microbiome feature classification
# Descrption: This notebook develops and evaluates machine learing models to classify patient samples into three clinical conditions (Alzheimers Diease, Mild Cognitive Impairment and Healthy).
# This is done using the microbiome derived features.
# The two feature sets we compared are:
# Relative Abundance Features: PCA summaries and top genera abundances
# Realtive Abundance + MICOM features: RA and MICOM-predicted reaction fluxes

# The pipeline includes the dataset loading, data cleaning, feature merging, model training, cross-validation, hyperparameter tuning, and classification performance reporting.
# The major classes/ features include:
# train_test_split which splits the dataset into stratified training and testing subsets
# Pipeline which combines preprocessing steps for standarization and estimators for SVM
# SVC which uses an RBF kernel for non-linear clinical seperation
# Random forest classifier which classifies using bagged decision trees for robustness and feature interaction learning
# Grid search which performs cross validated hyperparameter tuning to optimize SVM
# Classification report which computes precision, recall, F1 score and support for each class to assess the model performance

# The preprocessing libraries were used to prepare, process, and analyze the data:
# Pandas is used for data loading, feature subsetting and handling NaNs
# Numpy is used for numerical operations
# sklearn.preprocessing.StandardScaler is used for feature standardization in SVM
# sklearn.model_selection does train/ test splitting, cross validation and grid search
# sklearn.svm.SVC is the SVM classifier
# sklearn.ensemble.RandomForestClassifier is the RF classifier for baseline and RA + MICOM modeling
# sklearn.metrics.classification_report is used for the performance evaluation metrics
# seaborn / matplotlib is used for visualization and exploratory plots
# pathlib and os handle the file and project folder organization


Connect to Google Drive and Create MICOM Folder

In [ ]:
# Mount Google Drive
# This is needed to run the notebook in Google Colab and read the data stored in the drive.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Create main folder for project inputs and outputs
# This folder will store the shared datasets, MICOM output files, and features tables for machine learning algorithms

import os
ml_path = '/content/drive/MyDrive/Machine_Learning'
os.makedirs(ml_path, exist_ok=True)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#Pandas and numpy are needed for data structuring and mathematical operations
#Seaborn and matplotlib are used for visualization and exploratory data analysis


Import Feature Tables

In [ ]:
rel_abd_features = pd.read_csv(Path(ml_path) / "relative_abundance_features.csv")
micom_features = pd.read_csv(Path(ml_path) / "micom_features.csv")
#Loading the CSV file with microbiome relative abundances
#The micom dataset was previously generated using MICOM (microbiome metabolism modeling) features
#The dataframe in pandas can store many features, and so, we are isolating for only the microbial relative abundance features here

#The relative abundance features will contain data for microbial PCA components, top microbial genera with strongest inter-condition differences, and sample ID as well as clinical conditions
#The MICOM features will contain community-level reaction fluxes, metabolic exchange features, commmunity growth rate

In [ ]:
print(rel_abd_features)

     sample_id condition       PC1       PC2       PC3  Escherichia  \
0   SRR8061715        AD -2.420204  0.264267  2.695112     0.000000   
1   SRR8061716        AD -0.305117  0.802686  3.401735     0.509391   
2   SRR8061717        AD -0.655024 -0.822020  0.503503     0.294271   
3   SRR8061718        AD -0.517932  0.679205  3.005779     0.841240   
4   SRR8061719        AD -2.342568  1.186368  1.623629     0.000000   
..         ...       ...       ...       ...       ...          ...   
88  SRR8061803       MCI  2.245148 -0.389435  4.055000     0.178281   
89  SRR8061804       MCI  2.274504 -0.679650  1.003374     0.295258   
90  SRR8061805       MCI -0.002851  0.235047  1.655528     0.183633   
91  SRR8061806       MCI -2.346745 -0.576592 -1.549863     0.028533   
92  SRR8061807       MCI  1.069313 -1.520620 -0.468429     0.000000   

    Bacteroides   Blautia  Alistipes  Limosilactobacillus  
0      0.000000  0.061740   0.000000             0.563767  
1      0.030804  0.084899  

In [ ]:
# For all MICOM feature columns (i.e., all non-ID columns), replace NaN with 0.
id_cols = ["sample_id", "condition"]
micom_feature_cols = [c for c in micom_features.columns if c not in id_cols]
micom_features[micom_feature_cols] = micom_features[micom_feature_cols].fillna(0)
#NaNs values appear because some reactions and metabolites are not expressed in every microbiome community
#The ML algorithm cannot accept NaNs, so we have to replace missing values with zero to indicate that the reaction is negligable

# Rename MICOM condition column
#This is doen to avoid collision after merging
micom_features = micom_features.rename(columns={"condition": "condition_micom"})

In [ ]:
print(micom_features)
#This is done as a check to look through the MICOM table

     sample_id condition_micom  EX_xyluglc_m  EX_rpn_96990_glc_m     EX_ac_m  \
0   SRR8061716              AD      0.000000            0.000000  690.466643   
1   SRR8061715              AD      0.000000            0.131094   -0.000016   
2   SRR8061718              AD      0.000000            0.000000  878.207937   
3   SRR8061722              AD      0.000000            0.000000  919.346421   
4   SRR8061719              AD     -0.037830            0.080319    0.037646   
5   SRR8061732              AD     -0.000000            0.000000  642.309347   
6   SRR8061734              AD     -0.000000            0.000000  867.447321   
7   SRR8061743              AD      0.000000            0.000000  852.189339   
8   SRR8061771              AD     -7.593526           12.586949    9.424648   
9   SRR8061772              AD     -1.085796            2.056029   11.857900   
10  SRR8061777              AD      0.000000            6.711693    4.216986   
11  SRR8061737             MCI      0.00

In [ ]:
# Merge RA and MICOM feature tables, keeping only samples that appear in both (how="inner").
# Only inner samples are retained to ensure feature alignment
ra_micom_features = rel_abd_features.merge(micom_features, on="sample_id", how="inner")

# Remove extra condition column.
ra_micom_features = ra_micom_features.drop(columns=["condition_micom"])
#The following are the columns that are ready to be used by the model (numeric values except for the ID)
ra_micom_feature_cols = [c for c in ra_micom_features.columns if c not in id_cols]

print("Total Machine Learning Features: ", ra_micom_features.shape[1])
print("ML feature columns:", ra_micom_feature_cols)
print(ra_micom_features.head())

Total Machine Learning Features:  16
ML feature columns: ['PC1', 'PC2', 'PC3', 'Escherichia', 'Bacteroides', 'Blautia', 'Alistipes', 'Limosilactobacillus', 'EX_xyluglc_m', 'EX_rpn_96990_glc_m', 'EX_ac_m', 'EX_co2_m', 'EX_7ocholate_m', 'growth']
    sample_id condition       PC1       PC2       PC3  Escherichia  \
0  SRR8061715        AD -2.420204  0.264267  2.695112     0.000000   
1  SRR8061716        AD -0.305117  0.802686  3.401735     0.509391   
2  SRR8061718        AD -0.517932  0.679205  3.005779     0.841240   
3  SRR8061719        AD -2.342568  1.186368  1.623629     0.000000   
4  SRR8061722        AD -0.213649  0.991315  3.570985     0.905615   

   Bacteroides   Blautia  Alistipes  Limosilactobacillus  EX_xyluglc_m  \
0     0.000000  0.061740        0.0             0.563767       0.00000   
1     0.030804  0.084899        0.0             0.000000       0.00000   
2     0.000000  0.011743        0.0             0.020667       0.00000   
3     0.000000  0.105855        0.0   

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report
#Sklearn is used to split the data into train and test set, cross validate and hyperparameter tune and classify pipelines and scaling

Train Models with Only Relative Abundance Features

In [ ]:
# Define features (X) and ground truth labels (y).
# Features: PCA components (PC1, PC2, PC3) and top 5 genera based on inter-condition range
#X_ra keeps the low-dimensional PCA summaries of the microbial structures
# The top microbial genera iis then found based on the greatest variance between AD, MCI and Healthy
X_ra = rel_abd_features.drop(columns=["sample_id", "condition"])
# Ground-Truth Labels: AD, MCI, or Healthy. --> these indicate the cognitive states
y_ra = rel_abd_features["condition"]

In [ ]:
# Split into training (70%) and testing (30%) sets.
X_train_ra, X_test_ra, y_train_ra, y_test_ra = train_test_split(
    X_ra,
    y_ra,
    test_size = 0.3,
    random_state = 5,
    stratify = y_ra)    # same proportion of each condition in training/testing sets
    #stratification is important to preserve the class ratios

Model A: Support Vector Machine

In [ ]:
# Create Model A SVM pipeline
# Use Radial Basis Function (rbf) kernel because features likely have non-linear boundaries.
# This pipeline uses StandardScalar to normalize each feature to unit variance, which prevents imbalance
# The RBF kernel here functions as way to learn non-linear seperation between the clinical conditions
svm_ra = Pipeline([
    ("scaler", StandardScaler()),    # calculates mean and standard deviation from training set
    ("svm", SVC(kernel = "rbf", probability = True, random_state = 5))])

In [ ]:
# Default SVM
# Perform 5-fold cross-validation on training set to estimate performance (split training
# data into 5 groups, train on 4 groups, validate on remaining group).

default_model_a_cv = cross_val_score(svm_ra, X_train_ra, y_train_ra, cv = 5)

# Train the default model on the full training set.
svm_ra.fit(X_train_ra, y_train_ra)
# Use default model to classify unseen testing set samples.
default_model_a_pred = svm_ra.predict(X_test_ra)

print("[RA] Default SVM 5-fold CV accuracy:", default_model_a_cv.mean())
print("[RA] Default SVM classification report:")
print(classification_report(y_test_ra, default_model_a_pred))


[RA] Default SVM 5-fold CV accuracy: 0.5846153846153845
[RA] Default SVM classification report:
              precision    recall  f1-score   support

          AD       0.44      0.40      0.42        10
     Healthy       0.25      0.25      0.25         8
         MCI       0.36      0.40      0.38        10

    accuracy                           0.36        28
   macro avg       0.35      0.35      0.35        28
weighted avg       0.36      0.36      0.36        28



In [ ]:
# Hyperparameter Tuning
# Create dictionary for possible hyperparameter values.
# The grid search explores a combination of misclassification (C) and kernal curvature (gamma)
# C = regularization strength:
#    - Large C: strong penalty for misclassification (risk of overfitting).
#    - Small C: lenient model (risk of underfitting).
# gamma = influence of training samples:
#    - Large gamma: each sample has local influence (risk of overfitting).
#    - Small gamma: each sample has broad influence (risk of underfitting).
hyperparameters = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1]}

# Use GridSearchCV to evaluate model performance using different combinations of hyperparameters.
hyperparam_tuning = GridSearchCV(
    svm_ra,
    hyperparameters,
    cv = 5,
    n_jobs = -1)

# Fit grid search on the training set.
# Perform 5-fold cross-validation on training set to estimate performance and refits best model.
hyperparam_tuning.fit(X_train_ra, y_train_ra)

# Select the best performing SVM model.
best_svm_ra = hyperparam_tuning.best_estimator_
# Use tuned model to classify unseen testing set samples.
y_pred_svm_ra = best_svm_ra.predict(X_test_ra)

print("Tuned hyperparameters:", hyperparam_tuning.best_params_)
print("[RA] Tuned SVM 5-fold CV accuracy:", hyperparam_tuning.best_score_)
print("[RA] Tuned SVM classification report:")
print(classification_report(y_test_ra, y_pred_svm_ra))

Tuned hyperparameters: {'svm__C': 100, 'svm__gamma': 0.001}
[RA] Tuned SVM 5-fold CV accuracy: 0.6153846153846154
[RA] Tuned SVM classification report:
              precision    recall  f1-score   support

          AD       0.50      0.40      0.44        10
     Healthy       0.22      0.25      0.24         8
         MCI       0.36      0.40      0.38        10

    accuracy                           0.36        28
   macro avg       0.36      0.35      0.35        28
weighted avg       0.37      0.36      0.36        28



Model B: Random Forest

In [ ]:
# Default Random Forest model
# This is a decision tree of samples and class_weight = 'balanced' adjusts any misclassification penalty to handle a natural class imbalanace
rf_ra = RandomForestClassifier(
    n_estimators = 500,  # number of trees
    random_state = 5,
    n_jobs = -1,
    class_weight = "balanced")

# Perform 5-fold cross-validation on training set to estimate performance (split training
# data into 5 groups, train on 4 groups, validate on remaining group).
# This is an evaluation of the default random forest
default_model_b_cv = cross_val_score(rf_ra, X_train_ra, y_train_ra, cv = 5)
print("[RA] Default Random Forest 5-fold CV accuracy:", default_model_b_cv.mean())

# Train default model on the full training set.
rf_ra.fit(X_train_ra, y_train_ra)
# # Use default model to classify unseen testing set samples.
y_pred_rf_ra = rf_ra.predict(X_test_ra)

print("[RA] Default Random Forest classification report:")
print(classification_report(y_test_ra, y_pred_rf_ra))

[RA] Default Random Forest 5-fold CV accuracy: 0.5230769230769232
[RA] Default Random Forest classification report:
              precision    recall  f1-score   support

          AD       0.44      0.40      0.42        10
     Healthy       0.22      0.25      0.24         8
         MCI       0.40      0.40      0.40        10

    accuracy                           0.36        28
   macro avg       0.36      0.35      0.35        28
weighted avg       0.37      0.36      0.36        28



Train Models with Relative Abundance and MICOM Features

In [ ]:
# Define features (X) and ground truth labels (y).
# Features: PCA components, top 5 genera based on inter-condition range, top 5 reactions, community growth
X_rm = ra_micom_features.drop(columns=["sample_id", "condition"])
#X_rm contains the community PCA strcture, top microbial genera, top MICOM reaction fluxes and community growth rate
# Ground-Truth Labels: AD, MCI, or Healthy.
#y_rm is the clinical labels ^mentioned above
y_rm = ra_micom_features["condition"]

In [ ]:
# Split into training (70%) and testing (30%) sets.
X_train_rm, X_test_rm, y_train_rm, y_test_rm = train_test_split(
    X_rm,
    y_rm,
    test_size = 0.3,
    random_state = 15,
    stratify = y_rm)    # same proportion of each condition in training/testing sets

Model C: Support Vector Machine

In [ ]:
# Create Model C SVM pipeline
#Same architecture as before but now trained on expanded feature set
# Use Radial Basis Function (rbf) kernel because features likely have non-linear boundaries.
svm_rm = Pipeline([
    ("scaler", StandardScaler()),    # calculates mean and standard deviation from training set
    ("svm", SVC(kernel = "rbf", probability = True, random_state = 15))])

In [ ]:
# Default SVM
# Train the default model on the full training set.
svm_rm.fit(X_train_rm, y_train_rm)
# Use default model to classify unseen testing set samples.
default_model_c_pred = svm_rm.predict(X_test_rm)

print("[RA+MICOM] Default SVM classification report:")
print(classification_report(y_test_rm, default_model_c_pred))

[RA+MICOM] Default SVM classification report:
              precision    recall  f1-score   support

          AD       0.60      0.75      0.67         4
     Healthy       0.00      0.00      0.00         2
         MCI       0.67      1.00      0.80         2

    accuracy                           0.62         8
   macro avg       0.42      0.58      0.49         8
weighted avg       0.47      0.62      0.53         8



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Hyperparameter Tuning
# Create dictionary for possible hyperparameter values.
# C = regularization strength:
#    - Large C: strong penalty for misclassification (risk of overfitting).
#    - Small C: lenient model (risk of underfitting).
# gamma = influence of training samples:
#    - Large gamma: each sample has local influence (risk of overfitting).
#    - Small gamma: each sample has broad influence (risk of underfitting).
#This is the same parameter grid as earlier
hyperparameters = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1]}

# Use GridSearchCV to evaluate model performance using different combinations of hyperparameters.
rm_hyperparam_tuning = GridSearchCV(
    svm_rm,
    hyperparameters,
    cv = 5,
    n_jobs = -1)

# Fit grid search on the training set.
# Perform 5-fold cross-validation on training set to estimate performance and refits best model.
rm_hyperparam_tuning.fit(X_train_rm, y_train_rm)

# Select the best performing SVM model.
best_svm_rm = rm_hyperparam_tuning.best_estimator_
# Use tuned model to classify unseen testing set samples.
y_pred_svm_rm = best_svm_rm.predict(X_test_rm)

print("Tuned hyperparameters:", rm_hyperparam_tuning.best_params_)
print("[RA+MICOM] Tuned SVM 5-fold CV accuracy:", rm_hyperparam_tuning.best_score_)
print("[RA+MICOM] Tuned SVM classification report:")
print(classification_report(y_test_rm, y_pred_svm_rm))

Tuned hyperparameters: {'svm__C': 100, 'svm__gamma': 0.001}
[RA+MICOM] Tuned SVM 5-fold CV accuracy: 0.41666666666666663
[RA+MICOM] Tuned SVM classification report:
              precision    recall  f1-score   support

          AD       0.75      0.75      0.75         4
     Healthy       1.00      0.50      0.67         2
         MCI       0.67      1.00      0.80         2

    accuracy                           0.75         8
   macro avg       0.81      0.75      0.74         8
weighted avg       0.79      0.75      0.74         8



Model D: Random Forest

In [ ]:
# Default Random Forest model
rf_rm = RandomForestClassifier(
    n_estimators = 500,  # number of trees
    random_state = 15,
    n_jobs = -1,
    class_weight = "balanced")

# Train default model on the full training set.
rf_rm.fit(X_train_rm, y_train_rm)
# Use default model to classify unseen testing set samples.
y_pred_rf_rm = rf_rm.predict(X_test_rm)

print("[RA+MICOM] Default Random Forest classification report:")
print(classification_report(y_test_rm, y_pred_rf_rm))

[RA+MICOM] Default Random Forest classification report:
              precision    recall  f1-score   support

          AD       0.75      0.75      0.75         4
     Healthy       0.50      0.50      0.50         2
         MCI       0.50      0.50      0.50         2

    accuracy                           0.62         8
   macro avg       0.58      0.58      0.58         8
weighted avg       0.62      0.62      0.62         8

